# Protein Foundation Model Benchmark (85 HQ Proteins)

**Models benchmarked:**
- ESM-1b (sequence)
- ESM-3 (sequence)
- SaProt (structure-aware)
- AlphaGenome (DNA-based)
- Evo2 7B (DNA-based)

In [1]:
# Mount Google Drive (read-only access to stored data)
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip -q install biopython pandas numpy torch scikit-learn scipy tqdm umap-learn matplotlib seaborn requests transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.7 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ─── SOURCE DATA (READ-ONLY — do NOT write here) ───
DRIVE_BASE = Path("/content/drive/MyDrive/protein_benchmark_v2")
MANIFEST_DIR = DRIVE_BASE / "manifest"
FASTA_DIR = DRIVE_BASE / "fasta"
AF_DIR = DRIVE_BASE / "alphafold_structures"
RCSB_DIR = DRIVE_BASE / "rcsb_structures"
EMB_DIR = DRIVE_BASE / "embeddings"

# ─── OUTPUT (separate Drive folder — does NOT touch source data) ───
OUT_BASE = Path("/content/drive/MyDrive/protein_benchmark_v2_RESULTS")
OUT_TAB = OUT_BASE / "tables"
OUT_FIG = OUT_BASE / "figures"
OUT_EMB = OUT_BASE / "embeddings"
for d in [OUT_TAB, OUT_FIG, OUT_EMB]:
    d.mkdir(parents=True, exist_ok=True)

print("Source (Drive, read-only):", DRIVE_BASE)
print("Output (Drive, new folder):", OUT_BASE)

Device: cuda
Source (Drive, read-only): /content/drive/MyDrive/protein_benchmark_v2
Output (Drive, new folder): /content/drive/MyDrive/protein_benchmark_v2_RESULTS


In [4]:
# Load the 85-protein high-quality manifest
manifest_hq = pd.read_csv(MANIFEST_DIR / "manifest_hq.csv")
print(f"Loaded manifest_hq: {len(manifest_hq)} proteins")
print(f"Groups: {manifest_hq['group'].nunique()}")
print(manifest_hq["group"].value_counts())
manifest_hq.head()

Loaded manifest_hq: 85 proteins
Groups: 6
group
tubulin         24
pi3k            20
gpcr            18
kinase          11
zinc_binding    10
dna_binding      2
Name: count, dtype: int64


,acc,group,seq
0,P18074,dna_binding,MKLNVDGLLVYFPYDYIYPEQFSYMRELKRTLDAKGHGVLEMPSGT...
1,Q99551,dna_binding,MQSLSLGQTSISKGLNYLTIMAPGNLWHMRNNFLFGSRCWMTRFSA...
2,P41146,gpcr,MEPLFPAPFWEVIYGSHLQGNLSLLSPNHSLLPPHLLLNASHGAFL...
3,Q14416,gpcr,MGSLLALLALLLLWGAVAEGPAKKVLTLEGDLVLGGLFPVHQKGGP...
4,P29275,gpcr,MLLETQDALYVALELVIAALSVAGNVLVCAAVGTANTLQTPTNYFL...


## Load Pre-Computed Protein Embeddings (from Drive)

In [5]:
# Load protein model embeddings from Drive
esm1b_mean = torch.load(EMB_DIR / "esm1b_mean.pt", map_location="cpu")
esm3_mean = torch.load(EMB_DIR / "esm3_open_mean.pt", map_location="cpu")
saprot_mean = torch.load(EMB_DIR / "saprot_mean.pt", map_location="cpu")

print(f"ESM-1b embeddings: {len(esm1b_mean)}")
print(f"ESM-3 embeddings: {len(esm3_mean)}")
print(f"SaProt embeddings: {len(saprot_mean)}")

# Subset to HQ accessions
acc_hq = manifest_hq["acc"].tolist()

esm1b_hq = {a: esm1b_mean[a] for a in acc_hq if a in esm1b_mean}
esm3_hq = {a: esm3_mean[a] for a in acc_hq if a in esm3_mean}
saprot_hq = {a: saprot_mean[a] for a in acc_hq if a in saprot_mean}

print(f"\nHQ subset:")
print(f"  ESM-1b: {len(esm1b_hq)}")
print(f"  ESM-3: {len(esm3_hq)}")
print(f"  SaProt: {len(saprot_hq)}")

ESM-1b embeddings: 85
ESM-3 embeddings: 85
SaProt embeddings: 85

HQ subset:
  ESM-1b: 85
  ESM-3: 85
  SaProt: 85


## DNA-Based Model: AlphaGenome

AlphaGenome operates on DNA sequences. Since we have protein sequences, we reverse-translate
using the most common human codons, then pad/truncate to the model's expected input length (131,072 bp).

In [6]:
!pip -q install alphagenome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.9/176.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 35.4 MB/s eta 0:00:00


In [7]:
# Codon table (most frequent human codons)
CODON_TABLE = {
    'A': 'GCC', 'R': 'CGG', 'N': 'AAC', 'D': 'GAC', 'C': 'TGC',
    'E': 'GAG', 'Q': 'CAG', 'G': 'GGC', 'H': 'CAC', 'I': 'ATC',
    'L': 'CTG', 'K': 'AAG', 'M': 'ATG', 'F': 'TTC', 'P': 'CCC',
    'S': 'AGC', 'T': 'ACC', 'W': 'TGG', 'Y': 'TAC', 'V': 'GTG',
    '*': 'TGA',
}

def protein_to_dna(seq):
    codons = [CODON_TABLE.get(aa, 'NNN') for aa in seq.upper()]
    return ''.join(codons)

# Reverse-translate all proteins to DNA
dna_sequences = {}
for _, row in manifest_hq.iterrows():
    acc, seq = row["acc"], row["seq"]
    dna_sequences[acc] = protein_to_dna(seq)

print(f"DNA sequences generated: {len(dna_sequences)}")
sample_acc = list(dna_sequences.keys())[0]
print(f"Sample: {sample_acc}, protein len={len(manifest_hq[manifest_hq['acc']==sample_acc]['seq'].values[0])}, DNA len={len(dna_sequences[sample_acc])}")

DNA sequences generated: 85
Sample: P18074, protein len=760, DNA len=2280


In [11]:
# Verify DNA sequences look correct
sample_acc = list(dna_sequences.keys())[0]
sample_dna = dna_sequences[sample_acc]
print(f"Sample DNA (first 60 bases): {sample_dna[:60]}")
print(f"Total proteins with DNA: {len(dna_sequences)}")
print(f"Max DNA length: {max(len(s) for s in dna_sequences.values())}")
print(f"Min DNA length: {min(len(s) for s in dna_sequences.values())}")
print(f"AlphaGenome target length: {TARGET_LEN} (will be padded with Ns)")

Sample DNA (first 60 bases): ATGAAGCTGAACGTGGACGGCCTGCTGGTGTACTTCCCCTACGACTACATCTACCCCGAG
Total proteins with DNA: 85
Max DNA length: 3576
Min DNA length: 324
AlphaGenome target length: 131072 (will be padded with Ns)


In [14]:
import time
from alphagenome.models import dna_client
from alphagenome.models.dna_output import OutputType
from google.colab import userdata

API_KEY = userdata.get("ALPHA_GENOME_API_KEY")
client = dna_client.create(API_KEY)

def normalize_dna(seq, target_len=TARGET_LEN):
    seq = seq.upper()
    if len(seq) > target_len:
        start = (len(seq) - target_len) // 2
        return seq[start:start + target_len]
    if len(seq) < target_len:
        pad = target_len - len(seq)
        return seq + "N" * pad
    return seq

def alphagenome_embedding(output):
    vectors = []
    tracks = [
        output.cage,
        output.chip_histone,
        output.chip_tf,
        output.dnase,
        output.procap,
        output.rna_seq,
    ]
    for track in tracks:
        if track is None:
            continue
        arr = track.values
        center = arr.shape[1] // 2
        window = arr[:, center - 5:center + 5]
        pooled = window.mean(axis=1)
        vectors.append(pooled)
    if not vectors:
        return None
    embedding = np.concatenate(vectors)
    return torch.tensor(embedding, dtype=torch.float32)

In [15]:
# Generate AlphaGenome embeddings
cache_path = OUT_EMB / "alphagenome_mean.pt"

if cache_path.exists():
    alphagenome_mean = torch.load(cache_path, map_location="cpu")
    print(f"Loaded cached AlphaGenome embeddings: {len(alphagenome_mean)}")
else:
    alphagenome_mean = {}

# Run inference for any missing accessions
accs_to_run = [acc for acc in dna_sequences if acc not in alphagenome_mean]
print(f"AlphaGenome: {len(alphagenome_mean)} cached, {len(accs_to_run)} remaining")

BATCH_SIZE = 2
for i in tqdm(range(0, len(accs_to_run), BATCH_SIZE), desc="AlphaGenome"):
    batch_accs = accs_to_run[i:i + BATCH_SIZE]
    batch_seqs = [normalize_dna(dna_sequences[acc]) for acc in batch_accs]

    try:
        outputs = client.predict_sequences(
            sequences=batch_seqs,
            requested_outputs=[
                OutputType.CAGE,
                OutputType.CHIP_HISTONE,
                OutputType.CHIP_TF,
                OutputType.DNASE,
                OutputType.PROCAP,
                OutputType.RNA_SEQ,
            ],
            ontology_terms=[],
        )
        for acc, output in zip(batch_accs, outputs):
            emb = alphagenome_embedding(output)
            if emb is not None:
                alphagenome_mean[acc] = emb

        torch.save(alphagenome_mean, cache_path)
        time.sleep(10)

    except Exception as e:
        print(f"  Batch error: {e}")
        time.sleep(30)

torch.save(alphagenome_mean, cache_path)
print(f"AlphaGenome embeddings: {len(alphagenome_mean)}")

AlphaGenome: 0 cached, 85 remaining


AlphaGenome:   0%|          | 0/43 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

AlphaGenome embeddings: 85


## DNA-Based Model: Evo2 7B

Evo2 (Arc Institute) is a 7B-parameter genomic foundation model trained on DNA sequences.
We extract embeddings from an intermediate layer (block 28) and mean-pool over token positions.

In [ ]:
!pip -q install flash-attn==2.8.0.post2 --no-build-isolation
!pip -q install evo2

In [ ]:
from evo2 import Evo2

print("Loading Evo2 7B...")
evo2_model = Evo2("evo2_7b")
print("Evo2 7B loaded.")

In [ ]:
EVO2_LAYER = "blocks.28"

@torch.no_grad()
def evo2_embedding(seq):
    seq = seq.upper()
    input_ids = torch.tensor(
        evo2_model.tokenizer.tokenize(seq),
        dtype=torch.int,
    ).unsqueeze(0).to("cuda:0")

    _, embeddings = evo2_model(input_ids, return_embeddings=True, layer_names=[EVO2_LAYER])
    hidden = embeddings[EVO2_LAYER]  # (1, seq_len, 4096)
    emb = hidden.mean(dim=1).squeeze().cpu().float()
    return emb

In [ ]:
# Generate Evo2 embeddings
cache_path_evo2 = OUT_EMB / "evo2_7b_mean.pt"

if cache_path_evo2.exists():
    evo2_mean = torch.load(cache_path_evo2, map_location="cpu")
    print(f"Loaded cached Evo2 embeddings: {len(evo2_mean)}")
else:
    print("Running Evo2 inference...")
    evo2_mean = {}

    for acc, dna_seq in tqdm(dna_sequences.items(), desc="Evo2 embed"):
        try:
            emb = evo2_embedding(dna_seq)
            evo2_mean[acc] = emb
        except Exception as e:
            print(f"  Failed {acc}: {e}")
        torch.cuda.empty_cache()

    torch.save(evo2_mean, cache_path_evo2)
    print(f"Evo2 embeddings saved: {len(evo2_mean)}")

print(f"Evo2 7B: {len(evo2_mean)} embeddings, dim={next(iter(evo2_mean.values())).shape[0]}")

# Free memory
del evo2_model
torch.cuda.empty_cache()
import gc; gc.collect()

## Assemble All Models for Benchmarking

In [ ]:
# Subset DNA model embeddings to HQ set
alphagenome_hq = {a: alphagenome_mean[a] for a in acc_hq if a in alphagenome_mean}
evo2_hq = {a: evo2_mean[a] for a in acc_hq if a in evo2_mean}

models = {
    "ESM-1b": esm1b_hq,
    "ESM-3": esm3_hq,
    "SaProt": saprot_hq,
    "AlphaGenome": alphagenome_hq,
    "Evo2": evo2_hq,
}

for name, emb in models.items():
    if len(emb) > 0:
        print(f"  {name}: {len(emb)} proteins, dim={next(iter(emb.values())).shape[0]}")
    else:
        print(f"  {name}: 0 proteins (FAILED)")

## Evaluation Functions

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_sim
from scipy.stats import spearmanr

def embeddings_to_dataset(embedding_dict, manifest_df):
    X, y = [], []
    for _, row in manifest_df.iterrows():
        acc = row["acc"]
        if acc in embedding_dict:
            X.append(embedding_dict[acc].numpy())
            y.append(row["group"])
    return np.vstack(X), np.array(y)

def compute_spearman(embedding_dict, manifest_df):
    X, y = embeddings_to_dataset(embedding_dict, manifest_df)
    sim = sklearn_cosine_sim(X)
    label_sim = (y[:, None] == y[None, :]).astype(int)
    iu = np.triu_indices(len(y), k=1)
    corr, _ = spearmanr(sim[iu], label_sim[iu])
    return corr

def linear_probe_cv(embedding_dict, manifest_df, folds=3):
    X, y = embeddings_to_dataset(embedding_dict, manifest_df)
    if len(X) < folds:
        return np.nan
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    scores = []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        if X.shape[1] > 5000:
            n_comp = min(len(train_idx) - 2, 50)
            pipe = make_pipeline(StandardScaler(), PCA(n_components=n_comp),
                                 LogisticRegression(max_iter=2000, random_state=42))
        else:
            pipe = make_pipeline(StandardScaler(),
                                 LogisticRegression(max_iter=2000, random_state=42))
        pipe.fit(X_train, y_train)
        scores.append(accuracy_score(y_test, pipe.predict(X_test)))
    return np.mean(scores)

def knn_cv(embedding_dict, manifest_df, k=5, folds=3):
    X, y = embeddings_to_dataset(embedding_dict, manifest_df)
    if len(X) < folds:
        return np.nan
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    scores = []
    for train_idx, test_idx in skf.split(X, y):
        clf = KNeighborsClassifier(n_neighbors=k, metric="cosine")
        clf.fit(X[train_idx], y[train_idx])
        scores.append(accuracy_score(y[test_idx], clf.predict(X[test_idx])))
    return np.mean(scores)

## Classification Benchmark (Spearman, Linear Probe, kNN)

In [ ]:
results = []

for name, emb_dict in models.items():
    if len(emb_dict) == 0:
        print(f"  {name}: SKIPPED (no embeddings)")
        continue
    sp = compute_spearman(emb_dict, manifest_hq)
    lp = linear_probe_cv(emb_dict, manifest_hq)
    knn = knn_cv(emb_dict, manifest_hq)
    print(f"  {name}: Spearman={sp:.3f}, LP={lp:.3f}, kNN={knn:.3f}")
    results.append({"Model": name, "Spearman": sp, "LinearProbe": lp, "kNN": knn})

results_df = pd.DataFrame(results).sort_values("LinearProbe", ascending=False)
results_df.to_csv(OUT_TAB / "classification_results.csv", index=False)
display(results_df)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


  ESM-1b: Spearman=0.324, LP=0.824, kNN=0.706


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


  ESM-3: Spearman=0.358, LP=0.788, kNN=0.659


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


  SaProt: Spearman=0.358, LP=0.659, kNN=0.682


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


## UMAP Visualization

In [ ]:
import umap
import matplotlib.pyplot as plt

active_models = {k: v for k, v in models.items() if len(v) > 0}
n_models = len(active_models)
cols = min(3, n_models)
rows = (n_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
if n_models == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for idx, (name, emb_dict) in enumerate(active_models.items()):
    X, y = embeddings_to_dataset(emb_dict, manifest_hq)
    reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, metric="cosine", random_state=42)
    Z = reducer.fit_transform(X)

    ax = axes[idx]
    for label in np.unique(y):
        mask = y == label
        ax.scatter(Z[mask, 0], Z[mask, 1], label=label, alpha=0.7, s=30)
    ax.set_title(name, fontsize=14)
    ax.legend(fontsize=8)

# Hide unused axes
for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig(OUT_FIG / "umap_all_models.png", dpi=150, bbox_inches="tight")
plt.show()

## Radar Plot

In [ ]:
metrics = ["Spearman", "LinearProbe", "kNN"]
data = results_df.set_index("Model")[metrics]
normalized = (data - data.min()) / (data.max() - data.min() + 1e-8)

labels = metrics
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for model_name, row in normalized.iterrows():
    values = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, values, label=model_name, linewidth=2)
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_title("Model Comparison (Normalized)", fontsize=14, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(OUT_FIG / "radar_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## Model Similarity Heatmap

In [ ]:
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

def model_similarity(model_a_dict, model_b_dict, manifest_df):
    common = [a for a in manifest_df["acc"] if a in model_a_dict and a in model_b_dict]
    if len(common) < 3:
        return np.nan
    Xa = np.stack([model_a_dict[a].numpy() for a in common])
    Xb = np.stack([model_b_dict[a].numpy() for a in common])
    Sa = cosine_similarity(Xa).flatten()
    Sb = cosine_similarity(Xb).flatten()
    return np.corrcoef(Sa, Sb)[0, 1]

model_names = [k for k in models.keys() if len(models[k]) > 0]
sim_matrix = np.zeros((len(model_names), len(model_names)))

for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        sim_matrix[i, j] = model_similarity(models[m1], models[m2], manifest_hq)

plt.figure(figsize=(7, 6))
sns.heatmap(sim_matrix, annot=True, fmt=".3f", cmap="coolwarm",
            xticklabels=model_names, yticklabels=model_names)
plt.title("Representation Similarity Between Models")
plt.tight_layout()
plt.savefig(OUT_FIG / "model_similarity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## Per-Model Cosine Similarity Heatmaps

In [ ]:
def plot_similarity_heatmap(embedding_dict, manifest_df, title):
    X, labels = [], []
    for _, row in manifest_df.iterrows():
        if row["acc"] in embedding_dict:
            X.append(embedding_dict[row["acc"]].numpy())
            labels.append(row["group"])
    if len(X) == 0:
        print(f"  Skipping {title}: no embeddings")
        return
    X = np.vstack(X)
    sim = cosine_similarity(X)

    plt.figure(figsize=(8, 7))
    sns.heatmap(sim, cmap="viridis", xticklabels=False, yticklabels=False)
    plt.title(title)
    plt.xlabel("Proteins")
    plt.ylabel("Proteins")
    plt.tight_layout()
    plt.savefig(OUT_FIG / f"similarity_{title.replace(' ', '_').lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()

for name, emb in models.items():
    if len(emb) > 0:
        plot_similarity_heatmap(emb, manifest_hq, f"{name} Similarity")

## Sequence-Identity Control

In [ ]:
from Bio import pairwise2

accs = manifest_hq["acc"].tolist()
seqs = manifest_hq.set_index("acc").loc[accs, "seq"].tolist()

def seq_identity(a, b):
    aln = pairwise2.align.globalxx(a, b, one_alignment_only=True)[0]
    matches = sum(x == y for x, y in zip(aln.seqA, aln.seqB))
    return matches / max(len(a), len(b))

# Sample pairs (full pairwise is expensive for 85 proteins)
import itertools, random
all_pairs = list(itertools.combinations(range(len(accs)), 2))
random.seed(42)
sampled_pairs = random.sample(all_pairs, min(500, len(all_pairs)))

ids_list, groups_same = [], []
for i, j in tqdm(sampled_pairs, desc="Sequence identity"):
    ids_list.append(seq_identity(seqs[i], seqs[j]))
    groups_same.append(1 if manifest_hq.iloc[i]["group"] == manifest_hq.iloc[j]["group"] else 0)

plt.figure(figsize=(6, 4))
plt.scatter(ids_list, groups_same, s=10, alpha=0.5)
plt.xlabel("Sequence Identity")
plt.ylabel("Same Group (1=yes, 0=no)")
plt.title("Sequence Identity vs Functional Group Membership")
plt.tight_layout()
plt.savefig(OUT_FIG / "seqid_vs_group.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Mean identity (same group): {np.mean([ids_list[k] for k in range(len(ids_list)) if groups_same[k]==1]):.3f}")
print(f"Mean identity (diff group): {np.mean([ids_list[k] for k in range(len(ids_list)) if groups_same[k]==0]):.3f}")

## Structural Evaluation — Spearman vs RCSB Experimental Structures

In [ ]:
import requests
from Bio.PDB import PDBParser

def ca_distance_vector(pdb_path, bins=100):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", str(pdb_path))
    coords = []
    for atom in structure.get_atoms():
        if atom.get_name() == "CA":
            coords.append(atom.get_coord())
    coords = np.array(coords)
    if len(coords) < 10:
        return None
    dist_matrix = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    dists = dist_matrix[np.triu_indices(len(coords), k=1)]
    hist, _ = np.histogram(dists, bins=bins, range=(0, 50), density=True)
    return hist

# Check what's in RCSB_DIR on Drive
rcsb_files = list(RCSB_DIR.glob("*.pdb"))
print(f"RCSB structure files on Drive: {len(rcsb_files)}")

if len(rcsb_files) > 0:
    rcsb_stems = [f.stem for f in rcsb_files]
    print(f"Sample filenames: {rcsb_stems[:5]}")

In [ ]:
# Build structural embeddings from RCSB structures
# First check if rcsb_map.csv exists (maps UniProt acc -> PDB file path)
rcsb_map_path = MANIFEST_DIR / "rcsb_map.csv"
structure_embeddings = {}

if rcsb_map_path.exists():
    rcsb_df = pd.read_csv(rcsb_map_path)
    rcsb_map = dict(zip(rcsb_df["acc"], rcsb_df["pdb_path"]))
    print(f"Loaded rcsb_map: {len(rcsb_map)} entries")

    for acc in tqdm(acc_hq, desc="Structural embeddings"):
        if acc not in rcsb_map:
            continue
        pdb_path = Path(rcsb_map[acc])
        if not pdb_path.exists():
            continue
        vec = ca_distance_vector(pdb_path)
        if vec is not None:
            structure_embeddings[acc] = vec

elif len(rcsb_files) > 0:
    # Try direct accession-named files
    for acc in tqdm(acc_hq, desc="Structural embeddings"):
        pdb_path = RCSB_DIR / f"{acc}.pdb"
        if not pdb_path.exists():
            continue
        vec = ca_distance_vector(pdb_path)
        if vec is not None:
            structure_embeddings[acc] = vec

else:
    # Fallback: download from RCSB
    print("No RCSB structures on Drive. Downloading...")
    LOCAL_RCSB = OUT_BASE / "rcsb_local"
    LOCAL_RCSB.mkdir(parents=True, exist_ok=True)

    def uniprot_to_pdb(acc):
        url = "https://search.rcsb.org/rcsbsearch/v2/query"
        query = {
            "query": {"type": "terminal", "service": "text",
                      "parameters": {
                          "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                          "operator": "exact_match", "value": acc}},
            "return_type": "entry"
        }
        try:
            r = requests.post(url, json=query, timeout=15)
            if r.status_code == 200:
                return [res["identifier"] for res in r.json().get("result_set", [])]
        except:
            pass
        return []

    def download_pdb_local(pdb_id, out_dir):
        path = out_dir / f"{pdb_id}.pdb"
        if path.exists():
            return path
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            path.write_bytes(r.content)
            return path
        return None

    for acc in tqdm(acc_hq, desc="Downloading RCSB"):
        pdb_ids = uniprot_to_pdb(acc)
        if not pdb_ids:
            continue
        path = download_pdb_local(pdb_ids[0], LOCAL_RCSB)
        if path:
            vec = ca_distance_vector(path)
            if vec is not None:
                structure_embeddings[acc] = vec

print(f"\nStructural embeddings built: {len(structure_embeddings)} / {len(acc_hq)} proteins")

In [ ]:
def structural_spearman(model_embeddings, structure_embeddings):
    accs_common = list(set(model_embeddings.keys()) & set(structure_embeddings.keys()))
    if len(accs_common) < 3:
        return np.nan

    sim_model, sim_struct = [], []
    for i in range(len(accs_common)):
        for j in range(i + 1, len(accs_common)):
            a, b = accs_common[i], accs_common[j]
            v1 = model_embeddings[a].numpy() if isinstance(model_embeddings[a], torch.Tensor) else model_embeddings[a]
            v2 = model_embeddings[b].numpy() if isinstance(model_embeddings[b], torch.Tensor) else model_embeddings[b]
            s1, s2 = structure_embeddings[a], structure_embeddings[b]

            sim_m = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
            sim_s = np.dot(s1, s2) / (np.linalg.norm(s1) * np.linalg.norm(s2))
            sim_model.append(sim_m)
            sim_struct.append(sim_s)

    if len(sim_model) == 0:
        return np.nan
    return spearmanr(sim_model, sim_struct)[0]

struct_results = []
for name, emb_dict in models.items():
    if len(emb_dict) == 0:
        struct_results.append({"Model": name, "StructuralSpearman": np.nan})
        continue
    score = structural_spearman(emb_dict, structure_embeddings)
    struct_results.append({"Model": name, "StructuralSpearman": score})
    print(f"  {name}: StructuralSpearman = {score:.4f}" if not np.isnan(score) else f"  {name}: N/A")

struct_df = pd.DataFrame(struct_results)
struct_df.to_csv(OUT_TAB / "structural_spearman.csv", index=False)
display(struct_df)

## Final Results Summary

In [ ]:
final_results = results_df.merge(struct_df, on="Model", how="left")

styled = final_results.style.format({
    "Spearman": "{:.3f}",
    "LinearProbe": "{:.2%}",
    "kNN": "{:.2%}",
    "StructuralSpearman": "{:.3f}"
}).background_gradient(cmap="Greens", subset=["LinearProbe", "kNN"])

print("=" * 70)
print("FINAL BENCHMARK RESULTS")
print(f"Dataset: {len(manifest_hq)} high-confidence proteins (pLDDT >= 0.80)")
print(f"Classes: {manifest_hq['group'].nunique()} functional groups")
print("=" * 70)
display(styled)

final_results.to_csv(OUT_TAB / "final_benchmark_metrics.csv", index=False)
print(f"\nResults saved to: {OUT_TAB}")

## Statistical Significance

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.metrics import pairwise_distances

def per_protein_knn(embedding_dict, manifest_df, k=5):
    X, y = embeddings_to_dataset(embedding_dict, manifest_df)
    D = pairwise_distances(X, metric="cosine")
    np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, axis=1)[:, :k]
    correct = np.array([y[i] in y[nn[i]] for i in range(len(y))])
    return correct.astype(float)

active_model_names = [k for k in models.keys() if len(models[k]) > 0]
model_correct = {}
for name in active_model_names:
    model_correct[name] = per_protein_knn(models[name], manifest_hq)

# Friedman test (non-parametric repeated measures)
arrays = [model_correct[m] for m in active_model_names]
min_len = min(len(a) for a in arrays)
arrays = [a[:min_len] for a in arrays]

if len(arrays) >= 3 and min_len >= 5:
    stat, p_val = friedmanchisquare(*arrays)
    print(f"Friedman test across all models: chi2={stat:.3f}, p={p_val:.4e}")
else:
    print("Not enough models/data for Friedman test")

# Pairwise Wilcoxon between top two
sorted_models = results_df["Model"].tolist()
if len(sorted_models) >= 2:
    best, second = sorted_models[0], sorted_models[1]
    if best in model_correct and second in model_correct:
        min_l = min(len(model_correct[best]), len(model_correct[second]))
        stat, p = wilcoxon(model_correct[best][:min_l], model_correct[second][:min_l])
        print(f"Wilcoxon {best} vs {second}: stat={stat:.1f}, p={p:.4e}")

In [ ]:
print("\nAll output files:")
for f in sorted(OUT_BASE.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(OUT_BASE)}")